In [1]:
# from comet_ml import Experiment
from trainer import Trainer
from diffusion import Diffusion
# from unet import Unet
from diffusers.models.unet_2d_condition import UNet2DConditionModel
from diffusers import DDPMScheduler
from transformers import CLIPTokenizer, CLIPTextModel
import argparse
import wandb
from torchsummary import summary
import torch

/opt/conda/envs/idl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/envs/idl/lib/python3.9/site-packages/diffusers/models/unet_2d_condition.py:20: FutureWarning: `UNet2DConditionOutput` is deprecated and will be removed in version 0.29. Importing `UNet2DConditionOutput` from `diffusers.models.unet_2d_condition` is deprecated and this will be removed in a future version. Please use `from diffusers.models.unets.unet_2d_condition import UNet2DConditionOutput`, instead.
  deprecate("UNet2DConditionOutput", "0.29", deprecation_message)
/opt/conda/envs/idl/lib/python3.9/site-packages/diffusers/models/unet_2d_condition.py:25: FutureWarning: `UNet2DConditionModel` is deprecated and will be removed in version 0.29. Importing `UNet2DConditionModel` from `diffusers.models.unet_2d_condition` is deprecated

In [2]:
device = ("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
device

'cuda'

In [3]:
pretrained_model_name_or_path = "runwayml/stable-diffusion-v1-5"

In [4]:
model = UNet2DConditionModel.from_pretrained(
        pretrained_model_name_or_path, subfolder="unet", torch_dtype=torch.float32
    ).to(device)

diffusion = Diffusion(
    model,
    image_size=32,
    channels=32,
    timesteps=50,  # number of steps
).to(device)


In [18]:
with torch.no_grad():
    inp = torch.randn((1, 4, 32, 32)).to(device).type(torch.float16)
    noise = torch.randn((1, 4, 32, 32)).to(device).type(torch.float16)
    encoding = torch.randn(0, 999, 768).to(device).type(torch.float16)
    # out = model(noise, torch.randint(1, 10, (1, )).to(device), encoding)
    out = model(noise.to(torch.float16), torch.randint(1, 10, (1,)).long().to(device), encoding, return_dict=False)[0]


In [19]:
out.shape

torch.Size([1, 4, 32, 32])

In [ ]:
# accelerate launch diffusers/examples/text_to_image/train_text_to_image_lora_genai.py --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4" --dataset_name="polinaeterna/pokemon-blip-captions" --dataloader_num_workers=0 --resolution=512 --center_crop --random_flip --train_batch_size=1 --gradient_accumulation_steps=4 --max_train_steps=15000 --learning_rate=1e-04 --max_grad_norm=1 --lr_scheduler="cosine" --lr_warmup_steps=0 --output_dir="output/pokemon-lora" --report_to=wandb --checkpointing_steps=1000 --validation_prompt="A pokemon" --seed=1337

In [30]:
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable_params = total_params - trainable_params

    print(f"Total Parameters: **{total_params:,}**")
    print(f"Trainable Parameters: **{trainable_params:,}**")
    print(f"Non-trainable Parameters: **{non_trainable_params:,}**")

# Example usage:
# model = YourModel()
# count_parameters(model)


In [34]:
tokenizer = CLIPTokenizer.from_pretrained(pretrained_model_name_or_path, subfolder="tokenizer")

In [29]:
count_parameters(model)

Total Parameters: **859,520,964**
Trainable Parameters: **859,520,964**
Non-trainable Parameters: **0**


In [20]:
noise_scheduler = DDPMScheduler.from_pretrained(pretrained_model_name_or_path, subfolder="scheduler")
noise_scheduler.config

FrozenDict([('num_train_timesteps', 1000),
            ('beta_start', 0.00085),
            ('beta_end', 0.012),
            ('beta_schedule', 'scaled_linear'),
            ('trained_betas', None),
            ('variance_type', 'fixed_small'),
            ('clip_sample', False),
            ('prediction_type', 'epsilon'),
            ('thresholding', False),
            ('dynamic_thresholding_ratio', 0.995),
            ('clip_sample_range', 1.0),
            ('sample_max_value', 1.0),
            ('timestep_spacing', 'leading'),
            ('steps_offset', 1),
            ('rescale_betas_zero_snr', False),
            ('_use_default_values',
             ['timestep_spacing',
              'dynamic_thresholding_ratio',
              'sample_max_value',
              'rescale_betas_zero_snr',
              'prediction_type',
              'clip_sample_range',
              'variance_type',
              'thresholding']),
            ('_class_name', 'DDPMScheduler'),
            ('_diff

In [ ]:
# accelerate launch diffusers/examples/text_to_image/train_text_to_image_lora_genai.py \
#     --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4" \
#     --dataset_name="polinaeterna/pokemon-blip-captions" \
#     --dataloader_num_workers=0 \
#     --resolution=512 \
#     --center_crop \
#     --random_flip \
#     --train_batch_size=1 \
#     --gradient_accumulation_steps=4 \
#     --max_train_steps=15000 \
#     --learning_rate=1e-04 \
#     --max_grad_norm=1 \
#     --lr_scheduler="cosine" \
#     --lr_warmup_steps=0 \
#     --output_dir="output/pokemon-lora" \
#     --report_to=wandb \
#     --checkpointing_steps=1000 \
#     --validation_prompt="A pokemon" \
#     --seed=1337

In [4]:
from diffusers import StableDiffusionPipeline

model_id = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe = pipe.to("cuda")

# 2. Forward embeddings and negative embeddings through text encoder
prompt = 25 * "a photo of an astronaut riding a horse on mars"
max_length = pipe.tokenizer.model_max_length
max_length

Loading pipeline components...: 100%|██████████| 7/7 [00:01<00:00,  3.89it/s]


77

In [65]:
from compel import Compel, ReturnedEmbeddingsType
from diffusers import DiffusionPipeline
import torch


class LLM:
    def __init__(self):
        self.tokenizer = CLIPTokenizer.from_pretrained(pretrained_model_name_or_path, subfolder="tokenizer")
        self.text_encoder = CLIPTextModel.from_pretrained(pretrained_model_name_or_path, subfolder="text_encoder")

    def encode_batch_1(self, texts):
        """
        Encodes a batch of texts using the BertTokenizer.

        Args:
            texts (list of str): List of input texts to encode.

        Returns:
            torch.Tensor: Tensor of token ids for the encoded texts.
        """
        text_encoding = []
        for text in texts:
            text_tokens = self.tokenizer(
                text, return_tensors="pt"
            ).input_ids
            # print(f"text_tokens.shape = {text_tokens.shape}")
            encoder_hidden_states = self.text_encoder(
                    text_tokens, return_dict=False)[0]
            # print(f"encoder_hidden_states.shape = {encoder_hidden_states.shape}")
            text_encoding.append(encoder_hidden_states)
        ret = torch.cat(text_encoding, dim=1)
        # print(f"ret.shape = {ret.shape}")
        return ret

In [56]:
from compel import Compel
tokenizer = CLIPTokenizer.from_pretrained(pretrained_model_name_or_path, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(pretrained_model_name_or_path, subfolder="text_encoder")
compel = Compel(tokenizer=tokenizer, text_encoder=text_encoder)
embeds = compel('("a cat playing in the forest", "an impressionist oil painting").and()')

In [45]:
(embeds[0, 73, :] != embeds[0, 140, :]).any()

tensor(True)

In [49]:
embeds[0, 73, :10], embeds[0, 152, :10]

(tensor([-1.4330, -0.6558,  0.3928, -0.3701, -4.1685,  0.3087, -0.7238, -1.1895,
          1.0579,  0.9467]),
 tensor([-0.6613, -0.7404,  1.2553,  0.2065, -3.3953,  0.5141, -0.4178,  0.5303,
         -1.4020,  2.1452]))

In [66]:
llm = LLM()
llm.encode_batch_1(["green cat", "blue dog with a tail", "yellow baffoon"])

text_tokens.shape = torch.Size([1, 4])
encoder_hidden_states.shape = torch.Size([1, 4, 768])
text_tokens.shape = torch.Size([1, 7])
encoder_hidden_states.shape = torch.Size([1, 7, 768])
text_tokens.shape = torch.Size([1, 6])
encoder_hidden_states.shape = torch.Size([1, 6, 768])
ret.shape = torch.Size([1, 17, 768])


tensor([[[-0.3884,  0.0229, -0.0522,  ..., -0.4899, -0.3066,  0.0675],
         [-0.6045, -2.1100,  0.6977,  ..., -0.3181, -0.8722,  0.5462],
         [-1.8845, -0.2767,  0.8697,  ..., -1.5928, -0.7341, -1.2417],
         ...,
         [-0.3256,  1.1171,  0.5600,  ..., -0.5779,  2.3403,  0.2922],
         [-0.7212,  1.3397,  0.0551,  ..., -1.7938,  2.4515,  0.6443],
         [ 0.5712,  0.2307, -0.9092,  ..., -0.7626,  1.5487,  0.0640]]],
       grad_fn=<CatBackward0>)

In [1]:
import os
import sys
sys.path.append(os.pardir)
sys.path

['/home/aravi/idlproject/text-to-image-generation/diffusion',
 '/opt/conda/envs/idl/lib/python39.zip',
 '/opt/conda/envs/idl/lib/python3.9',
 '/opt/conda/envs/idl/lib/python3.9/lib-dynload',
 '',
 '/opt/conda/envs/idl/lib/python3.9/site-packages',
 '..']

In [2]:
from ImageEncoder.latentdiffusion.image_encoding_helper import get_model, reconstruct_images

/opt/conda/envs/idl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ae_model = get_model('../ImageEncoder/latentdiffusion/models/first_stage_models/vq-f4/config.yaml', 'ae_model.ckpt')

/opt/conda/envs/idl/lib/python3.9/site-packages/torch/cuda/__init__.py:628: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 3, 64, 64) = 12288 dimensions.
making attention of type 'vanilla' with 512 in_channels


/opt/conda/envs/idl/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/envs/idl/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth
VQLPIPSWithDiscriminator running with hinge loss.
Restored from ae_model.ckpt with 0 missing and 0 unexpected keys
Restored from ae_model.ckpt with 0 missing and 0 unexpected keys


In [7]:
class LLM:
    def __init__(self):
        self.tokenizer = CLIPTokenizer.from_pretrained(pretrained_model_name_or_path, subfolder="tokenizer")
        self.text_encoder = CLIPTextModel.from_pretrained(pretrained_model_name_or_path, subfolder="text_encoder")

    def encode_batch_1(self, texts):
        """
        Encodes a batch of texts using the BertTokenizer.

        Args:
            texts (list of str): List of input texts to encode.

        Returns:
            torch.Tensor: Tensor of token ids for the encoded texts.
        """
        print(texts)
        text_encoding = []
        for text in texts:
            text_tokens = self.tokenizer(
                text, return_tensors="pt"
            ).input_ids
            print(text_tokens.shape)
            # print(f"text_tokens.shape = {text_tokens.shape}")
            encoder_hidden_states = self.text_encoder(
                    text_tokens, return_dict=False)[0]
            # print(f"encoder_hidden_states.shape = {encoder_hidden_states.shape}")
            text_encoding.append(encoder_hidden_states)
        ret = torch.cat(text_encoding, dim=1)
        # print(f"ret.shape = {ret.shape}")
        return ret

In [8]:
obj = LLM()
obj.encode_batch_1(["This person is wearing necktie. He has goatee. He is chubby. This person has big nose, and big lips. He is bald. He is wearing necktie. He has beard. The man has bags under eyes. He is wearing necktie. He is bald. He is wearing necktie. He has goatee, and big lips. He is chubby."])

Token indices sequence length is longer than the specified maximum sequence length for this model (78 > 77). Running this sequence through the model will result in indexing errors


['This person is wearing necktie. He has goatee. He is chubby. This person has big nose, and big lips. He is bald. He is wearing necktie. He has beard. The man has bags under eyes. He is wearing necktie. He is bald. He is wearing necktie. He has goatee, and big lips. He is chubby.']
torch.Size([1, 78])


RuntimeError: The size of tensor a (78) must match the size of tensor b (77) at non-singleton dimension 1

In [10]:
a = "This person is wearing necktie. He has goatee. He is chubby. This person has big nose, and big lips. He is bald. He is wearing necktie. He has beard. The man has bags under eyes. He is wearing necktie. He is bald. He is wearing necktie. He has goatee, and big lips. He is chubby."

In [11]:
text_tokens = obj.tokenizer(
                a, return_tensors="pt"
            ).input_ids
text_tokens.shape

torch.Size([1, 78])

: 